# 01 · Scraping

Descarga desde iNaturalist a `Insectos/<especie>/` y registra la procedencia en
`metadata.csv`. Incremental: deduplica por `photo_id`, así que reejecutar solo
baja lo que falta.

Solo licencias abiertas. Quedan fuera las fotos sin licencia declarada y
`cc-by-nc-nd`, cuyo ND prohíbe obras derivadas — recortar y aumentar lo es.

In [ ]:
import hashlib
import json
import re
import time

import requests

from corpus import IMAGENES, LISTA_ESPECIES, METADATA, escribirCsv, leerCsv

API = "https://api.inaturalist.org/v1"
LICENCIAS = "cc0,cc-by,cc-by-nc,cc-by-sa,cc-by-nc-sa"
LICENCIAS_OK = set(LICENCIAS.split(","))
PAUSA = 0.25          # entre descargas de fotos
PAUSA_API = 1.0       # entre llamadas a la API

POR_ESPECIE = 350
SOLO_ESPECIES = None          # {"eurybia_dardus"} para rellenar unas pocas
INCLUIR_NO_CONFIRMADAS = False

COLUMNAS = ["especie", "taxon_id", "observation_id", "photo_id", "quality_grade",
            "licencia", "atribucion", "url", "archivo", "sha256"]

## Cliente

In [23]:
def pedir(sesion, ruta, params, intentos=5):
    """Reintenta con espera creciente ante 429 y cortes de red."""
    for intento in range(intentos):
        try:
            respuesta = sesion.get(f"{API}/{ruta}", params=params, timeout=30)
        except requests.RequestException as e:
            print(f"    {type(e).__name__}, reintento en {2 ** intento}s")
            time.sleep(2 ** intento)
            continue
        if respuesta.status_code == 429:
            espera = float(respuesta.headers.get("Retry-After", 2 ** intento))
            print(f"    429, esperando {espera:.0f}s")
            time.sleep(espera)
            continue
        respuesta.raise_for_status()
        return respuesta.json()
    raise RuntimeError(f"{ruta} falló tras {intentos} intentos")


def nombreCorto(entrada):
    limpio = re.sub(r"\(.*?\)|,.*$", "", entrada)
    return " ".join(limpio.split()[:2])


def carpetaDe(nombre):
    return nombre.lower().replace(" ", "_")


def buscarTaxon(sesion, nombre):
    resultados = pedir(sesion, "taxa", {"q": nombre, "rank": "species",
                                        "per_page": 1, "is_active": "true"})["results"]
    return resultados[0] if resultados else None


def disponibilidad(sesion, taxonId):
    """Techo real de una especie, para saber si subir el tope sirve de algo."""
    return pedir(sesion, "observations", {
        "taxon_id": taxonId, "quality_grade": "research", "photos": "true",
        "photo_license": LICENCIAS, "per_page": 1,
    })["total_results"]


def descargar(sesion, url, destino):
    respuesta = sesion.get(url, timeout=45)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    return hashlib.sha256(respuesta.content).hexdigest()

## Paginación

`incluirNoConfirmadas` quita el filtro `quality_grade`: es la única fuente que
hace crecer a las especies escasas, que ya agotaron lo research-grade, a cambio
de identificaciones sin confirmar. Cada foto sale con su grado para poder
separarlas después.

In [24]:
def fotosDe(sesion, taxonId, maximo, incluirNoConfirmadas=False):
    vistas, pagina = set(), 1
    while len(vistas) < maximo:
        filtros = {"taxon_id": taxonId, "photos": "true", "photo_license": LICENCIAS,
                   "order_by": "id", "order": "asc", "per_page": 200, "page": pagina}
        if not incluirNoConfirmadas:
            filtros["quality_grade"] = "research"

        observaciones = pedir(sesion, "observations", filtros)["results"]
        if not observaciones:
            return
        for observacion in observaciones:
            for foto in observacion["photos"]:
                # el filtro del servidor es por observación: una válida arrastra
                # sus otras fotos, que pueden ser ND
                if foto["id"] in vistas or foto.get("license_code") not in LICENCIAS_OK:
                    continue
                vistas.add(foto["id"])
                yield observacion["id"], observacion.get("quality_grade", ""), foto
                if len(vistas) >= maximo:
                    return
        pagina += 1

## Descarga

Acumula sobre `metadata.csv` y lo reescribe entero al final

In [25]:
especies = json.loads(LISTA_ESPECIES.read_text())["species"]
especies = [e for e in especies if "sp." not in e["name"]]
if SOLO_ESPECIES:
    especies = [e for e in especies if carpetaDe(nombreCorto(e["name"])) in SOLO_ESPECIES]

sesion = requests.Session()
sesion.headers["User-Agent"] = "butterfly-dataset/1.0"

filas = leerCsv(METADATA)
caidas = []
conocidas = {fila["photo_id"] for fila in filas}
print(f"{len(filas)} fotos ya registradas · {len(especies)} especies a recorrer")

for numero, entrada in enumerate(especies, start=1):
    nombre = nombreCorto(entrada["name"])
    taxon = buscarTaxon(sesion, nombre)
    time.sleep(PAUSA)
    if taxon is None:
        print(f"[{numero}/{len(especies)}] {nombre}: sin taxón")
        continue

    carpeta = IMAGENES / carpetaDe(nombre)
    carpeta.mkdir(parents=True, exist_ok=True)
    nuevas = 0
    for observacionId, grado, foto in fotosDe(sesion, taxon["id"], POR_ESPECIE,
                                              INCLUIR_NO_CONFIRMADAS):
        if str(foto["id"]) in conocidas:
            continue
        url = foto["url"].replace("square", "large")
        archivo = carpeta / f"{carpetaDe(nombre)}_{foto['id']}.jpg"
        try:
            sha = descargar(sesion, url, archivo)
        except requests.RequestException as e:   # una foto caída no debe abortar la corrida
            caidas.append({"archivo": archivo.name, "url": url, "error": str(e)[:80]})
            time.sleep(PAUSA)
            continue
        filas.append({
            "especie": carpetaDe(nombre), "taxon_id": taxon["id"],
            "observation_id": observacionId, "photo_id": foto["id"],
            "quality_grade": grado, "licencia": foto["license_code"],
            "atribucion": foto.get("attribution", ""), "url": url,
            "archivo": archivo.name, "sha256": sha,
        })
        nuevas += 1
        time.sleep(PAUSA)

    escribirCsv(METADATA, filas, COLUMNAS)   # por especie: interrumpir no deja imágenes sin procedencia
    print(f"[{numero}/{len(especies)}] {nombre}: +{nuevas}")

print(f"\n{len(filas)} fotos en metadata.csv · {len(caidas)} descargas fallidas")

20038 fotos ya registradas · 86 especies a recorrer
[1/86] Pythonides jovianus: +0
[2/86] Panoquina ocola: +0
[3/86] Vettius fantasos: +0
[4/86] Pyrgus adepta: +0
[5/86] Pyrgus orcus: +0
[6/86] Urbanus elmina: +0
[7/86] Urbanus procne: +0
[8/86] Urbanus dorantes: +0
[9/86] Urbanus simplicius: +0
[10/86] Urbanus teleus: +0
[11/86] Autochton itylus: +0
[12/86] Parides eurimedes: +0
[13/86] Parides erithalion: +0
[14/86] Eurema arbela: +0
[15/86] Phoebis sennae: +0
[16/86] Eurema albula: +0
[17/86] Eurema xantochlora: +0
[18/86] Pyrisitia limbia: sin taxón
[19/86] Pyrisitia proterpia: +0
[20/86] Dismorphia amphione: +0
[21/86] Dismorphia thermesia: +0
[22/86] Dismorphia theucharila: +0
[23/86] Ascia monuste: +0
[24/86] Ectima thecla: +0
[25/86] Hamadryas februa: +0
[26/86] Nica flavilla: +0
[27/86] Hamadryas amphinome: +0
[28/86] Hamadryas feronia: +0
[29/86] Consul fabius: +0
[30/86] Hypna clytemnestra: +0
[31/86] Memphis pseudiphis: +0
[32/86] Siderone galanthis: +0
[33/86] Archaeoprepo

## Inventario

In [26]:
import pandas as pd

meta = pd.DataFrame(filas).drop_duplicates("photo_id")
porEspecie = meta.groupby("especie").agg(fotos=("photo_id", "size"),
                                         observaciones=("observation_id", "nunique"))
print(f"{len(porEspecie)} especies · {porEspecie.fotos.sum():,} fotos")
print(f"por especie: min {porEspecie.fotos.min()}, mediana "
      f"{porEspecie.fotos.median():.0f}, max {porEspecie.fotos.max()}")
porEspecie.sort_values("fotos").head(10)

84 especies · 23,515 fotos
por especie: min 15, mediana 350, max 350


,fotos,observaciones
especie,,
urbanus_elmina,15,10
dircenna_jemina,29,18
eurybia_dardus,39,23
memphis_pseudiphis,43,33
pseudodebis_puritana,47,40
taygetis_sylvia,53,45
cissia_confusa,54,45
pyrgus_adepta,56,31
vettius_fantasos,57,35


## Cuánto más hay disponible

Antes de subir el tope conviene saber el techo: una especie que ya agotó
iNaturalist no crece por más que se scrapee.

In [31]:
techos = {}
for especie, fila in porEspecie.iterrows():
    taxon = meta[meta.especie == especie].taxon_id.iloc[0]
    techos[especie] = disponibilidad(sesion, int(taxon))
    time.sleep(PAUSA_API)

porEspecie["obs_disponibles"] = pd.Series(techos)
porEspecie["fotos_estimadas"] = (porEspecie.obs_disponibles
                                 * porEspecie.fotos / porEspecie.observaciones).round()
for tope in (200, 350, 500):
    n = (porEspecie.fotos_estimadas >= tope).sum()
    print(f"  alcanzarían {tope} fotos: {n}/{len(porEspecie)} especies")
porEspecie.nsmallest(10, "fotos_estimadas")

  alcanzarían 200 fotos: 65/84 especies
  alcanzarían 350 fotos: 57/84 especies
  alcanzarían 500 fotos: 52/84 especies


,fotos,observaciones,obs_disponibles,fotos_estimadas
especie,,,,
urbanus_elmina,15,10,10,15.0
dircenna_jemina,29,18,18,29.0
eurybia_dardus,39,23,23,39.0
memphis_pseudiphis,43,33,33,43.0
pseudodebis_puritana,47,40,40,47.0
taygetis_sylvia,53,45,45,53.0
cissia_confusa,54,45,45,54.0
pyrgus_adepta,56,31,31,56.0
vettius_fantasos,57,35,35,57.0
